<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/hf-llm-01-wip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hugging Face NLP Course - Part 1 🤗📚

This notebook summarizes my learnings from the [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course/), covering chapters 1 to 4. 📚

## Setup 🛠️

Install the necessary libraries for this notebook: 📚💻

In [ ]:
%pip install python-dotenv
%pip install transformers[sentencepiece]
%pip install scikit-learn scipy

To access all models, you need an access token 🗝️ with the **Make calls to the serverless Inference API** permission. Create one [here](https://huggingface.co/settings/tokens) 🌐 and set it as the `HF_TOKEN` environment variable. 🖥️

Load the environment variables 🌍 and check that `HF_TOKEN` is available 🔑:

In [ ]:
import os

# Load environment variables from .env (if available)
from dotenv import load_dotenv
load_dotenv()

# If running in Google Colab, copy
# required secrets into environment variables
try:
  from google.colab import userdata
  os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except:
  pass

assert os.getenv("HF_TOKEN"), "You need to set the HF_TOKEN environment variable to run this notebook"

Now, let's check if PyTorch can access an NVIDIA GPU 🖥️. While not mandatory, using a GPU significantly speeds up model training ⚡. This check ensures CUDA is installed and accessible by PyTorch ✅. To skip GPU usage, simply comment out the cell below: 📝

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Warning: No TPU or GPU detected. Using CPU.")

print(f"🔥 Device Selected: {DEVICE}")

If you got no errors, you're ready to go! 🚀📚

## Tasks 📝✨

Let's explore the `transformers` library 🔍 to load pre-trained models 🤖 for various tasks. ✨ The `pipeline` abstraction in `transformers` provides easy access to various transformer models for different tasks 🎯, automatically selecting the best model 🏆.

### Sentiment analysis 😊📊

Classify the sentiment of one or more sentences using the default model `distilbert/distilbert-base-uncased-finetuned-sst-2-english` for sentiment analysis: 😊📊

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", device=DEVICE)
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
        "It hurts so good!",
        "While the service was certainly unique, it left a lasting impression that I won’t forget anytime soon."
    ]
)

The model correctly labeled each sentence as positive 😊 or negative 😞.

### Zero-shot classification 🤖✨

In a `zero-shot-classification` task, a model classifies the probability of an input matching provided, non-predefined labels: 📊🤖✨

In [ ]:
classifier = pipeline("zero-shot-classification", device=DEVICE)
classifier(
    "The ball curved beautifully into the top corner, leaving the goalkeeper with no chance.",
    candidate_labels=["sports", "art", "technology", "cooking", "nature"]
)

The model classified the sentence as primarily about `sports` 🏅, with possible links to `technology` ⚙️ (physics of a moving ball) or `art` 🎨 ("curved beautifully"), but excluded `nature` 🌳 and `cooking` 🍳.

### Text Generation ✍️✨

In a `text-generation` task ✍️, the model adds tokens ➕ to the right ➡️ of the given sentence.

In [ ]:
generator = pipeline("text-generation", device=DEVICE)
generator(
    "In this course, we will teach you how to",
    max_length=30, # Generate a sentence with a maximum length of 30 tokens
    num_return_sequences=2, # Generate 2 candidate sentence completions with the provided input as the prefix
)

Notice how the model generates coherent sentences 📝 based on our input. ✍️

### Mask Filling 🎭✨

The `mask-filling` task involves predicting the most likely tokens for the `<mask>` token's location: 🧩🔍

In [ ]:
unmasker = pipeline("fill-mask", device=DEVICE)
unmasker("This course will teach you all about <mask> models.", top_k=2)

The model correctly predicted the missing token, classifying `mathematical` 📊 as the most likely choice ✅ based on the sentence context.

### Named Entity Recognition 🏷️🔍

The `ner` task identifies and tags important words in the text by category (e.g., names 🧑‍🤝‍🧑, locations 📍).

In [ ]:
ner = pipeline("ner", grouped_entities=True, device=DEVICE)
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

The model detected a person's name (Sylvain) 👤, an organization (Hugging Face) 🏢, and a location (Brooklyn) 📍, pinpointing their exact positions in the text and indicating its confidence in the classifications.

### Question Answering ❓🤔

With the `question-answering` pipeline, the user inputs a context 📚 and a question ❓, and the model answers based on the context: 💡

In [ ]:
question_answerer = pipeline("question-answering", device=DEVICE)
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

The model correctly answered the question ✅ using the provided context 📚.

### Summarization 📚✨

The `summarization` pipeline generates a summary 📄 of the input text ✍️:

In [ ]:
summarizer = pipeline("summarization", device=DEVICE)
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)


### Translation

The `translation` pipeline translates text between languages 🌍✍️. A model must be explicitly selected to define the source and target languages 🔄:

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en", device=DEVICE)
translator("Ce cours est produit par Hugging Face.")

Observe how the model successfully translated the text from French 🇫🇷 to English 🇬🇧.

### Feature Extraction 🔍✨

The `feature-extraction` pipeline converts sentences into embeddings: 📝➡️🔍✨

In [ ]:
feature_extractor = pipeline("feature-extraction", device=DEVICE)
result = feature_extractor([
    "Woodpecker",
    "How much wood would a woodchuck chuck if a woodchuck could chuck wood?",
    "Soccer"
])
result

The model converts each sentence into a `768`-dimensional vector 📏, useful for clustering 📊, classification 🗂️, or dimensionality reduction 📉. By comparing the cosine similarity 🔍 of two vectors, we can assess the similarity of the corresponding sentences 📝.

## Using Transformers ⚡🤖

Transformers are neural network architectures that efficiently process and generate text using *self-attention* to understand word relationships. They can be categorized into three main types:

**GPT-like Models (Auto-Regressive Transformers)**  
- Models like GPT (Generative Pre-trained Transformer) are designed for **causal language modeling**, predicting the next word based on previous ones. 📝  
- They are **decoder-only models**, ideal for text generation tasks like story writing and chatbots. 💬  

**BERT-like Models (Auto-Encoding Transformers)**  
- BERT (Bidirectional Encoder Representations from Transformers) uses **masked language modeling (MLM)** to predict missing words in a sentence. 🔍  
- As **encoder-only models**, they focus on understanding input rather than generating text. 📖  
- These models excel in tasks requiring deep understanding, such as sentence classification, named entity recognition (NER), and question answering. ✅  

**BART/T5-like Models (Sequence-to-Sequence Transformers)**  
- These models combine encoder and decoder architectures, making them **encoder-decoder models** (sequence-to-sequence). 🔄  
- They are designed for **generative tasks requiring input**, such as machine translation, text summarization, and text-based question answering. 🌐  

Each Transformer type is optimized for different tasks, enhancing versatility in various natural language processing (NLP) applications. 🌟

### Biases 🤐

All models have biases that must be recognized in real-world applications ⚖️. For example, `bert-base-uncased` shows biased predictions for male and female professions 👨‍💼👩‍💼:

In [ ]:
unmasker = pipeline("fill-mask", model="bert-base-uncased", device=DEVICE)
[x["token_str"] for x in unmasker("This man works as a [MASK].")], [x["token_str"] for x in unmasker("This woman works as a [MASK].")]

No comments. 🤐

### Behind the Pipeline 🔍💧

Let's manually set up the tasks behind the `pipeline` abstraction. Below is a `sentiment-analysis` task using the `distilbert/distilbert-base-uncased-finetuned-sst-2-english` model, while still utilizing the `pipeline` abstraction: 📊✨

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device=DEVICE)
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

To set up the task manually, load the **model** 🏗️ and **tokenizer**, preprocess the input text 📄, and pass it through the model. Each model has a specific tokenizer 🔑 that converts text into tokens. The `AutoTokenizer` class automatically selects the correct tokenizer for a model ✅:

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer

Now that we have the tokenizer 🧩, we can preprocess the input text 📄 by tokenizing it and converting it to input IDs for the model. We will pad sequences to a maximum length 📏, truncate overly long sequences ✂️, and return the token IDs as PyTorch tensors 🔢:

In [ ]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(
    raw_inputs, # The texts to tokenize
    padding=True, # Pad the inputs to the maximum input length
    truncation=True, # Truncate the text to the maximum length the model can accept
    return_tensors="pt" # Return PyTorch tensors
).to(DEVICE)
(
    inputs.keys(),
    inputs['input_ids'].shape,
    inputs["input_ids"],
    inputs['attention_mask'].shape,
    inputs["attention_mask"]
)

The tokenizer returns a dictionary with two keys: `input_ids`, which holds the tokenized inputs 📝, and `attention_mask`, a binary mask indicating padding in `input_ids` 🛡️. The model ignores padding during predictions 🚫. Both tensors have the shape `(2, 16)`, representing the batch size (two sentences) 📄📄 and maximum sequence length (16 tokens) 🔢.

To retrieve the model, use the `AutoModel` class: 📦✨

In [ ]:
from transformers import AutoModel

model_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(model_checkpoint).to(DEVICE)
model

With the model loaded 📦 and sentences tokenized 📝, we can now process our inputs through the model: 🚀

In [ ]:
model(**inputs)

The code above is equivalent to: 🔄

In [ ]:
outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
outputs

Notice however that the output doesn't have any logits. This is because when you load a model with `AutoModel` it just loads the base model without a head. For our classification task, we need to load the model with a classification head using `AutoModelForSequenceClassification`: 📦✨

In [ ]:
from transformers import AutoModelForSequenceClassification

model_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint).to(DEVICE)
outputs = model(**inputs)
outputs

The model outputs logits for two sentences across two classes (positive 😊 and negative 😞 sentiment). We can apply the `softmax` function to convert these logits into probabilities:

In [ ]:
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

We can inspect predicted labels using the model's config 🛠️📊:

In [ ]:
labels = model.config.id2label
labels

And use this to print the probability of each sentence being positive 😊 or negative 😞:

In [ ]:
for i, sentence in enumerate(raw_inputs):
    print("\n" + sentence)
    for index, prediction in enumerate(predictions[i]):
        label = labels[index]
        print(f"{index} ({label}): {prediction.item() * 100.0:.2f}%")

`AutoModel` automatically instantiates the correct model for the specified checkpoint 🗂️, but you can also directly instantiate specific models 🛠️:

In [ ]:
from transformers import BertConfig, BertModel

config = BertConfig()
model = BertModel(config).to(DEVICE)
model

Let's also inspect the model `config` 🔍:

In [ ]:
config

The example above shows a blank model 🏗️, but we can also load a pre-trained model 📦:

In [ ]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-cased").to(DEVICE)
model

Let's run tokenized sentences through the model: 🏃‍♂️💬🧠

In [ ]:
model(torch.tensor([
    [101, 7592, 999, 102], # "Hello!"
    [101, 4658, 1012, 102], # "Cool."
    [101, 3835, 999, 102], # "Nice!"
]).to(DEVICE))

### Tokenizers 🏷️✨

Tokenization divides text into smaller units, crucial for natural language processing (**NLP**). The three main types are **word-based, character-based, and subword-based**:

- **Word-Based Tokenization**  
  - Splits text by spaces or punctuation, assigning each word an ID. ✍️  
  - Requires a **large vocabulary** (~500K words in English). 📚  
  - Struggles with unknown words, represented as `[UNK]` or `<unk>`. ❓  

- **Character-Based Tokenization**  
  - Assigns an ID to each character, resulting in a **small vocabulary** and fewer unknown tokens. 🔤  
  - Increases input size and loses meaning compared to words. 📏  
  - Useful for languages like **Chinese**, where characters have significant meaning. 🇨🇳  

- **Subword-Based Tokenization**  
  - Combines word and character tokenization by keeping common words whole and splitting rarer ones into meaningful parts. 🔗  
  - Example: **"modernization"** can be split into **"modern"** and **"ization"**, aiding models in understanding word structures while managing vocabulary size. 🏗️  

- **Common Tokenization Strategies**  
  - **Byte Pair Encoding (BPE)** – Used in **GPT-2**, efficient for multilingual text. 🌐  
  - **WordPiece** – Used in **BERT**, enhances deep learning tokenization. 🧠  
  - **SentencePiece & Unigram** – Common in **multilingual models**, accommodating diverse scripts. 📝  

Each method addresses different NLP needs, balancing vocabulary size, efficiency, and meaning preservation. ⚖️

To load a tokenizer in `transformers` 🛠️, you can use the specific model architecture class 🏗️, just like with models:

In [ ]:
from transformers import BertTokenizer

BertTokenizer.from_pretrained("bert-base-cased")

Use the `AutoTokenizer` class 🛠️ to automatically select the correct tokenizer for a model 📚:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

To tokenize a sentence ✍️, split it into tokens 🔤 and encode them into token IDs 🔢. Calling the tokenizer on a sentence returns a dictionary 📚 with token IDs and an attention mask 🎭:

In [ ]:
tokenizer("Using a Transformer network is simple")

You can also perform each step manually: 🛠️✨

In [ ]:
tokens = tokenizer.tokenize("Using a Transformer network is simple")
tokens

With the sentence split into "subword" tokens 📝, we can encode them into token IDs 🔢 that the model understands 🤖:

In [ ]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

You can decode these token IDs back into text: 🔑📜

In [ ]:
tokenizer.decode(ids)

You can pass the IDs to the model as follows: 📥✨

In [ ]:
model(torch.tensor([ids]).to(DEVICE))

Remember, this won't work ❌: all sequences must be the same length 📏.

In [ ]:
try:
    model(torch.tensor([
        [200, 200, 200],
        [200, 200] # sequence length is different
    ]).to(DEVICE))
except Exception as e:
    print(e)

When sequences differ in length, pad them to match. 📏✨

In [ ]:
model(torch.tensor([
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id]
]).to(DEVICE))

Notice the following: 📢✨

In [ ]:
(
    model(torch.tensor([[200, 200]]).to(DEVICE)),
    model(torch.tensor([[200, 200, tokenizer.pad_token_id]]).to(DEVICE))
)

Notice how predictions change with the padding token? 🤔 The model considers it part of the input, altering predictions. 🔄 To prevent this, use the `attention_mask` from the tokenizer to instruct the model to ignore padding tokens: 🚫

In [ ]:
(
    model(
        torch.tensor([[200, 200]]).to(DEVICE),
        attention_mask=torch.tensor([[1, 1]]).to(DEVICE)
    ),
    model(
        torch.tensor([[200, 200, tokenizer.pad_token_id]]).to(DEVICE),
        attention_mask=torch.tensor([[1, 1, 0]]).to(DEVICE)
    )
)

Both predictions are identical 🔄 despite the input sequences having different lengths 📏.

### Putting it all together 🧩✨

To recap, pass a single sentence to the tokenizer: ✍️📜

In [ ]:
sequence = "I've been waiting for a HuggingFace course my whole life."
tokenizer(sequence)

You can pass a list of sentences: 📝✨

In [ ]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]
tokenizer(sequences)

Pad to the longest sequence in the batch: 📏✨

In [ ]:
tokenizer(sequences, padding="longest")

Pad to the model's maximum sequence length: 📏✨

In [ ]:
tokenizer(sequences, padding="max_length")

Padding should match the model's maximum length 📏, but sequences cannot exceed `N` tokens 🚫. If sequences are shorter than `N` tokens, they will be padded to `N` ➕; if longer, they will be truncated ✂️:

In [ ]:
tokenizer(sequences, padding="max_length", max_length=8)

When calling the tokenizer directly, it tokenizes the input ✍️ and adds special tokens like `[CLS]` 🔖 and `[SEP]` 🔖:

In [ ]:
sequence = "I've been waiting for a HuggingFace course my whole life."
model_inputs = tokenizer(sequence).to(DEVICE)
tokenizer.decode(model_inputs["input_ids"])

This won't occur if you tokenize ✂️ and encode 🔒 the input separately:

In [ ]:
tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
tokenizer.decode(ids)

### Fine-tuning a Pretrained Model 🔧🤖

Here’s how to fine-tune an existing model, let's first load a batch of sequences to be classified: 🔧✨

In [ ]:
import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification

# Retrieve the model and tokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(DEVICE)

# Tokenize two sentences
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(DEVICE)
batch

The tokenizer provided sequences for the model 📊. By adding expected labels 🏷️, we can use this batch structure to fine-tune the model 🔧:

In [ ]:
# Add labels to the batch (both sentences are positive)
batch["labels"] = torch.tensor([1, 1]).to(DEVICE)

# Create an instance of AdamW optimizer
optimizer = AdamW(model.parameters())

You can now run the cell below to perform an optimization step on the batch, if you run it multiple times, the loss should go down: 🔧📉

In [ ]:
# Forward pass the batch through the model and retrieve the calculated loss
loss = model(**batch).loss

# Backward pass to calculate the gradients
loss.backward()

# Perform a single optimization step to update the model's parameters based on the calculated gradients
optimizer.step()

# Print loss
loss

We can't achieve much with a small fine-tuning dataset, so let's retrieve a larger one using Hugging Face's `datasets` library 📚. First, we need to install it: 🛠️

In [ ]:
%pip install datasets

Now, we retrieve the [`MRPC`](https://paperswithcode.com/dataset/mrpc) dataset 📊, which contains sentence pairs 🗣️ labeled as paraphrases 🔄 or not:

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

The dataset is divided into `training`, `validation`, and `test` sets 📊, each containing two sentences ✍️✍️ and a label 🏷️ indicating if they are paraphrases. Let's examine the dataset features: 🔍

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset.features

Let's retrieve an example from the training set: 📚✨

In [ ]:
raw_train_dataset[0]

To train on this dataset 📊, each pair of sentences can be tokenized as a single sentence ✍️, separated by the `[SEP]` token 🔗, which the tokenizer does by default.

In [ ]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

Tokenize a batch of sentence pairs: 📝🔄

In [ ]:
inputs = tokenizer([
    ["This is the first sentence.", "This is the second one."],
    ["The first sentence is this one.", "The second sentence is this one."]
], padding=True)
torch.tensor(inputs["input_ids"]).shape, inputs

The tokenizer produced a batch of two tokenized sentences 📜✍️, which we need to feed as pairs to obtain the probability of them being paraphrases 🤔🔍. Let's examine one of these sentences:

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

At first glance, the batch appeared correct ✅, but it incorrectly paired the sentences ❌. Feeding two input lists into the tokenizer instead 🥪 treats it as a sentence pair task, causing it to tokenize corresponding items from both lists as a single sentence 📜:

In [ ]:
inputs = tokenizer(
    ["This is the first sentence.", "This is the second one."],
    ["The first sentence is this one.", "The second sentence is this one."]
, padding=True)
torch.tensor(inputs["input_ids"]).shape, inputs, tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

We now know how to tokenize the entire training dataset 📊✨:

In [ ]:
tokenized_dataset = tokenizer(
    raw_datasets["train"]["sentence1"],
    raw_datasets["train"]["sentence2"],
    padding=True,
    truncation=True,
    return_tensors="pt"
)
input_ids = tokenized_dataset["input_ids"]
input_ids.shape

We have an input batch of `3668` sequences 📊, each of length `103` 📏 (padded to the maximum sequence length).

The above tokenization works for small datasets, but for larger ones, the `map` function is preferable, as it allows processing batches in parallel without having to preload the dataset into memory first:

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True # Send multiple sentences to tokenize_function each call for better performance
)
tokenized_datasets

Notice that the tokenized dataset now includes new features from the tokenizer, such as `input_ids` 🆔 and `attention_mask` 🎭.

Let's investigate the sequence lengths of the tokenized training set: 📊✨

In [ ]:
len(set([len(x) for x in tokenized_datasets["train"]["input_ids"]])) # Count number of unique sequence lengths

If the tokenized dataset was padded to the maximum sequence length, all sequences would be the same length, which is not the case. ❌ This is expected, as padding is only necessary when feeding sequences to the model, and it should match the batch length, not the dataset length. 📏 We need to use a `DataCollator` to handle this: 🛠️

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator

Create a sample batch of tokenized sequences: 📝✨

In [ ]:
samples = tokenized_datasets["train"][:8] # Pick first 8 samples
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]} # Remove unnecessary columns
samples.keys()

Feed the batch into the data collator 📊, which will pad the sequences 📏 to the length of the longest one. 🏆

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

Notice that all sequences are now the same length of `67` 📏, which is the length of the longest sequence in the batch 📊. Let's confirm ✅:

In [ ]:
max([len(x) for x in tokenized_datasets["train"][:8]["input_ids"]])

Correct! ✅ Let's set up our data from scratch: load the dataset 📂, tokenize it ✂️, and feed it to the data collator 📊:

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# Load the tokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Load dataset
raw_datasets = load_dataset("glue", "mrpc")

# Tokenize dataset
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Wrap the tokenized dataset with the DataCollatorWithPadding
# (to make sure batches have same sequence length)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Load the model for fine-tuning 🔧 and specify the number of labels to predict 📊:

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(DEVICE)
model

The warning above indicates that the loaded model lacked a classification head for two labels ⚠️, so a new one with random weights was added ✨. We will utilize the pre-trained weights from the model 💪, but the classification head will be trained from scratch 🛠️:

Now, we initialize the trainer: 🎓✨

In [ ]:
from transformers import Trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    "test-trainer",
    bf16=True # Enable mixed precision training
)
trainer = Trainer(
    model, # the instantiated 🤗 Transformers model to be trained
    training_args, # training arguments, defined above
    train_dataset=tokenized_datasets["train"], # The dataset to train the model on
    eval_dataset=tokenized_datasets["validation"], # The dataset to evaluate the model on
    data_collator=data_collator, # defaults to DataCollatorWithPadding if not provided
    tokenizer=tokenizer # The tokenizer to be used,
)
training_args, trainer

We can now train the model: 🏋️‍♂️📊

In [ ]:
trainer.train()

Training is complete ✅, and loss has significantly decreased 📉. We can now evaluate the model on the validation set 🧪 using the trainer's `predict()` function, which simplifies the process by handling tokenization automatically 🔄:

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])
predictions.predictions.shape, predictions.label_ids.shape

The resulting object contains a `predictions` tensor with `2` logits for the predicted labels of each of the `408` validation sequences 📊, and a `label_ids` tensor with the ground truth label for each sequence 📜.

Let's check the logits 📊, predictions 📈, and ground truth labels ✅ for some of the results:

In [ ]:
import numpy as np

[(predictions.predictions[x], np.argmax(predictions.predictions[x], axis=-1), predictions.label_ids[x]) for x in range(3)]

Now, let's calculate the total number of accurate predictions for the entire validation dataset 📊:

In [ ]:
predicted_labels = np.argmax(predictions.predictions, axis=-1)
predicted_labels

We can now compare with the ground truth labels: 📊🔍

In [ ]:
np.sum(predicted_labels == predictions.label_ids) / len(predicted_labels)

Our model accurately predicts labels for **>80%** 📊 of the validation set.

A better way to evaluate performance is to use `evaluate` 📊, which considers dataset characteristics 📈 and applies the most suitable metrics ✅. First let's install it:

In [ ]:
%pip install evaluate

Let's now compute the metrics for our predictions. 📊📈

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=predicted_labels, references=predictions.label_ids)

If we create a method that takes a tuple of logits 📊 and ground truth labels 🏷️ and returns evaluation metrics 📈, we can provide it to the `Trainer`:

In [ ]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
compute_metrics((predictions.predictions, predictions.label_ids))

Let's pass `compute_metrics` to `Trainer` and try again 📊:

In [ ]:
training_args = TrainingArguments(
    "test-trainer",
    evaluation_strategy="epoch",
    fp16=True
)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(DEVICE)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

The training process now reports  validation loss 📉 due to the `compute_metrics` method we provided. We can see the training loss decreasing while validation loss increases, which means the model is overfitting 📉.

Now let's perform the same training run without the `Trainer` class 📚✨:

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets["train"].column_names

Now that we've tokenized the sentences, we can remove `sentence1` and `sentence2` from the dataset. 📊 Additionally, `idx` is unnecessary, and `label` should be renamed to `labels` for compatibility with Hugging Face:

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

Now that the dataset is ready, we can create a `DataCollator` to pad sequences 📏 and create `DataLoader` objects to load batches:

In [ ]:
from torch.utils.data import DataLoader

# Create data collator to pad batch sequences
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create a data loader to load batches from the training set
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True, # Shuffle the training set
    batch_size=8, # Each batch will have 8 samples
    collate_fn=data_collator # Use the data collator to pad the samples in the batch
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=8, # Each batch will have 8 samples
    collate_fn=data_collator # Use the data collator to pad the samples in the batch
)

Let's load a batch from the training set and inspect its contents: 📊✨

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

Each batch has `8` sequences 📊, each padded to the maximum sequence length of the batch 📏 (may be different with each batch).

Let's create the model and run the batch through it:

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load the pretrained model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(DEVICE)

# Send batch to GPU
batch.to(DEVICE)

# Forward pass the batch through the model
outputs = model(**batch)

(outputs.loss, outputs.logits.shape)

Running the batch through the model resulted in a logits tensor of shape `(8, 2)`, representing the strength for each class (paraphrase or not) for each of the `8` sequences 📊. Since the batch contains the ground-truth labels, a loss was computed as well, representing how close the predictions represented by the logits matches the ground-truth.

Let's now run training without using `Trainer`:

In [ ]:
from tqdm.auto import tqdm
from transformers import get_scheduler
from transformers import AdamW

# Create an instance of AdamW optimizer with starting learning rate of 5e-5
optimizer = AdamW(model.parameters(), lr=5e-5)

# Calculate the number of training steps
# (num_batches * n_epochs)
num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)

# Create learning rate scheduler
lr_scheduler = get_scheduler(
    "linear", # Linearly decrease the learning rate
    optimizer=optimizer, # The optimizer to use
    num_warmup_steps=0, # No warmup steps
    num_training_steps=num_training_steps # Total number of training steps
)

# Set the model in training mode (this will
# make sure that the model tracks gradients)
model.train()

# Create a progress bar where 100% = num_training_steps
progress_bar = tqdm(range(num_training_steps))

# Train for N epochs
for epoch in range(num_epochs):
    # Sample a batch from the training set
    for batch in train_dataloader:
        # Move batch to GPU
        batch.to(DEVICE)

        # Forward pass batch through the model
        outputs = model(**batch)

        # Backpropagate the loss through
        # the model (gradient calculation)
        loss = outputs.loss
        loss.backward()

        # Perform a single optimization step
        optimizer.step()

        # Update the learning rate using the linear scheduler
        lr_scheduler.step()

        # Zero out the gradients for the next batch
        # (otherwise they would accumulate)
        optimizer.zero_grad()

        # Update the progress bar
        progress_bar.update(1)

# Run last batch through model and output logits
outputs = model(**batch)
outputs.logits

Let's use the `evaluate` package to load the dataset metrics and calculate them for the model we just trained:

In [ ]:
import evaluate

# Change the model to evaluation mode
# (remove dropout layers, change batch norm layers to eval mode, etc.)
model.eval()

# Load metrics from dataset
metric = evaluate.load("glue", "mrpc")

# Run inference on the evaluation set and add predictions to the metric
for batch in eval_dataloader:
    # Move batch to GPU
    batch.to(DEVICE)

    # Forward pass the batch through the model
    # (disable gradient calculation to speed up computation)
    with torch.no_grad(): outputs = model(**batch)

    # Calculate predictions from logits
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Add batch to metric
    metric.add_batch(predictions=predictions, references=batch["labels"])

# Compute final metrics
metric.compute()

**Now with `"accelerate"` 🚀:** Hugging Face’s `accelerate` **automates multi-GPU, TPU, and mixed precision training**, boosting speed and reducing memory use. It removes the need for `DataParallel`, optimizes device placement, and enables **seamless FP16/BF16 training**—just wrap your model with `accelerator.prepare()`, and you're set! 🚀🔥

In [ ]:
from accelerate import Accelerator
from transformers import AdamW, AutoModelForSequenceClassification, get_scheduler

# Load the pre-trained model, add classification head
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Create an instance of AdamW optimizer
optimizer = AdamW(model.parameters(), lr=3e-5)

# Prepare for training using the Accelerator
accelerator = Accelerator()
train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

# Create the learning rate scheduler
num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

# Set the model in training mode
# (eg: enable dropout layers, etc.)
model.train()

# Train the model for N epochs
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    for batch in train_dl:
        # Forward pass batch through the model
        outputs = model(**batch)

        # Calculate the loss and perform a backward pass
        loss = outputs.loss
        accelerator.backward(loss)

        # Perform a single optimization step
        # (updates weights using gradients calculated during backpropagation)
        optimizer.step()

        # Perform a learning rate step
        lr_scheduler.step()

        # Zero out the gradients for the next batch
        # (otherwise they would accumulate)
        optimizer.zero_grad()

        # Update the progress bar
        progress_bar.update(1)

# Run last batch through model and output logits
outputs = model(**batch)
outputs.logits